**1. Imports**

Standard libraries: numpy/pandas for data handling, matplotlib/seaborn for plotting (not actually used yet — left over from early exploration), and the specific sklearn pieces needed later for splitting, preprocessing, the model, and evaluation metrics.

In [43]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

**2. Load the CSV**

pd.read_csv(...) reads your raw import records into df. Everything downstream depends on this being the completed dataset (importation-v1(COMPLETED).csv).

In [44]:
df = pd.read_csv('importation-v1(COMPLETED).csv')

**3. df.head()**

Just a visual sanity check on the first few rows — confirms the load worked and the columns look as expected before doing anything else.

In [45]:
df.head()

,year,port,country,HS,weight-kg,worth-usd,HS-Short-Name,distance,delivery-days,s-cost,tariff,VAT,Tariff-Amount-USD,VAT-Amount-USD,Total-Import-Cost-USD,Profit-Margin-Pct,Cost-per-kg-USD,Profit-per-kg-USD,Total-Import-Profit-USD
0,1403,shaeed rajaii,China,98870315,11179099,176091909,Vehicle/Chassis/Tractor Parts,10500,30,1676864.85,40%,9%,70436763.6,22187580.53,270393118.0,25%,24.19,6.05,67598279.50
1,1403,shaeed rajaii,China,98871182,20632931,162439389,Vehicle/Chassis/Tractor Parts,10500,30,3094939.65,40%,9%,64975755.6,20467363.01,250977447.3,25%,12.16,3.04,62744361.82
2,1403,shaeed rajaii,China,98871183,20275247,140716265,Vehicle/Chassis/Tractor Parts,10500,30,3041287.05,40%,9%,56286506.0,17730249.39,217774307.4,25%,10.74,2.68,54443576.86
3,1403,shaeed rajaii,China,87032290,14735380,124830304,Passenger Vehicles,10500,30,2210307.00,45%,9%,56173636.8,16290354.67,199504602.5,25%,13.54,3.38,49876150.62
4,1403,shaeed rajaii,China,98870121,14375399,113825856,Vehicle/Chassis/Tractor Parts,10500,30,2156309.85,40%,9%,45530342.4,14342057.86,175854566.1,25%,12.23,3.06,43963641.53


**4. Drop leakage & irrelevant columns**

Removes columns that either aren't part of your feature set (year, port, distance, delivery-days, s-cost) or are leakage — dollar amounts mathematically derived from profit (Tariff-Amount-USD, VAT-Amount-USD, Total-Import-Cost-USD, Cost-per-kg-USD, Profit-per-kg-USD, Profit-Margin-Pct). Leakage columns would let the model "cheat" by seeing numbers computed from the answer it's supposed to predict, so they're dropped before any modeling happens.

In [46]:
df.drop(columns=['year', 'port', 'HS', 'Profit-Margin-Pct', 'Profit-per-kg-USD',
                 'distance', 'delivery-days', 's-cost', 'Tariff-Amount-USD',
                 'VAT-Amount-USD', 'Total-Import-Cost-USD', 'Cost-per-kg-USD'], inplace=True)

**5. Check for missing values**

df.isna().sum() — confirms there's no missing data hiding in the remaining columns that would need handling before training.

In [47]:
df.isna().sum()

country                    0
weight-kg                  0
worth-usd                  0
HS-Short-Name              0
tariff                     0
VAT                        0
Total-Import-Profit-USD    0
dtype: int64

**6. First profit_rate check — finding the bad row**

Creates profit_rate (Total-Import-Profit-USD ÷ worth-usd) and immediately inspects it with .describe() and the top/bottom 10 values. This is diagnostic, not the final version of the column — it's how you originally discovered row 7112 had a profit_rate of 2.6 while everything else was clustered around 0.3, which led to investigating that row directly.

In [48]:
df['profit_rate'] = df['Total-Import-Profit-USD'] / df['worth-usd']
df['profit_rate'].describe()
df['profit_rate'].sort_values().tail(10)
df['profit_rate'].sort_values(ascending=False).head(10)

7112    2.607000
3212    0.609263
6344    0.587860
6731    0.579023
2336    0.549271
5985    0.541742
5920    0.539189
6607    0.533681
4577    0.522465
6967    0.496932
Name: profit_rate, dtype: float64

**7. Drop the bad row (7112)**

df.loc[7112] (checked separately, not shown as its own cell here) revealed weight-kg: 3800 against worth-usd: 50 — an impossible price of about 1.3 cents/kg. That's a data entry error, not a real outlier, so the row is dropped. This single row was responsible for unstable cross-validation scores later in the notebook — dropping it took your CV R² from an inconsistent [-1.74 … 0.88] range to a tight [0.82 … 0.95].

In [49]:
df = df.drop(index=7112)

**8. Convert tariff/VAT from percent-strings to decimals**

tariff/VAT arrived as text like "12%". .str.rstrip('%').astype(float) / 100 strips the %, converts to a number, then divides by 100 to get an actual decimal (0.12) the model can use as a plain numeric feature.

In [50]:
df['tariff'] = df['tariff'].str.rstrip('%').astype(float) / 100
df['VAT'] = df['VAT'].str.rstrip('%').astype(float) / 100
df.head()

,country,weight-kg,worth-usd,HS-Short-Name,tariff,VAT,Total-Import-Profit-USD,profit_rate
0,China,11179099,176091909,Vehicle/Chassis/Tractor Parts,0.40,0.09,67598279.50,0.383881
1,China,20632931,162439389,Vehicle/Chassis/Tractor Parts,0.40,0.09,62744361.82,0.386263
2,China,20275247,140716265,Vehicle/Chassis/Tractor Parts,0.40,0.09,54443576.86,0.386903
3,China,14735380,124830304,Passenger Vehicles,0.45,0.09,49876150.62,0.399552
4,China,14375399,113825856,Vehicle/Chassis/Tractor Parts,0.40,0.09,43963641.53,0.386236


**9. Confirm worth-usd has no zeros**

Checks .min() and counts exact zeros before dividing by worth-usd to build profit_rate — guards against a divide-by-zero producing inf/NaN values that would silently corrupt training.

In [51]:
df['worth-usd'].min()
(df['worth-usd'] == 0).sum()

np.int64(0)

**10. Recreate profit_rate (redundant — safe to remove)**

This recomputes the exact same formula as cell 6. Dropping a row doesn't change the calculation for the rows that remain, so this cell produces an identical result to cell 6 for every surviving row. Harmless, but it's duplicate work left over from iterating — cell 6 already gives you what you need once row 7112 is gone.

In [52]:
df['profit_rate'] = df['Total-Import-Profit-USD'] / df['worth-usd']
df['profit_rate'].describe()

count    7112.000000
mean        0.329020
std         0.018302
min         0.286572
25%         0.316379
50%         0.327684
75%         0.335412
max         0.609263
Name: profit_rate, dtype: float64

**11. Drop the raw profit column**

Total-Import-Profit-USD is removed now that profit_rate exists in its place as the prediction target. Keeping the raw dollar column around would be leakage — the model must never see the exact number it's trying to predict a rate from.

In [53]:
df.drop(columns=['Total-Import-Profit-USD'], inplace=True)

**12. Define feature groups**

Splits your six input columns into categorical_features (country, HS-Short-Name — text categories) and numerical_features (weight-kg, worth-usd, tariff, VAT — plain numbers). This grouping feeds directly into the preprocessor in the next cell.

In [54]:
categorical_features = ['country', 'HS-Short-Name']
numerical_features = ['weight-kg', 'worth-usd', 'tariff', 'VAT']

**13. Build the preprocessor**

ColumnTransformer passes numeric features through unchanged (tree-based models don't need scaling) and one-hot encodes the two categoricals — turning text categories into 0/1 columns the model can use. handle_unknown='ignore' means an unseen category at prediction time won't crash the pipeline, it just gets treated as all-zeros.

In [55]:
preprocessor = ColumnTransformer(transformers=[
    ('num', 'passthrough', numerical_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
])

**14. Define the model pipeline**

Pipeline chains the preprocessor and a RandomForestRegressor together so .fit()/.predict() can be called on raw, unprocessed data in one step. Not trained yet at this point — just defined.

In [56]:
pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                           ('regressor', RandomForestRegressor(random_state=42))])

**15. Split features and target**

x_data = the six model inputs. y_data = profit_rate, the target. Keeping these as the only two variables from df at this stage prevents any accidental leakage from slipping into the feature set.

In [57]:
x_data = df[['country', 'HS-Short-Name', 'weight-kg', 'worth-usd', 'tariff', 'VAT']]
y_data = df['profit_rate']

**16. Train/test split**

80/20 split via train_test_split, random_state=42 for reproducibility. The shape printout is a quick confirmation you're working with the row/column counts you expect.

In [58]:
train_x, test_x, train_y, test_y = train_test_split(x_data, y_data, test_size=0.2, random_state=42)
print("X_train shape:", train_x.shape)
print("X_test shape:", test_x.shape)

X_train shape: (5689, 6)
X_test shape: (1423, 6)


**17. Train, predict, evaluate, cross-validate**

pipeline.fit() trains the model on the 80% training split. r2_score/mean_absolute_error measure accuracy on the held-out 20%. The KFold(shuffle=True) + cross_val_score block re-splits the entire dataset five different ways and re-tests each — this is what originally caught row 7112, since one particular fold kept scoring badly until it was removed. shuffle=True matters because without it, KFold takes contiguous chunks — risky if rows in the CSV happen to be grouped by country.

In [59]:
pipeline.fit(train_x, train_y)
predictions = pipeline.predict(test_x)
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(pipeline, x_data, y_data, cv=kf, scoring='r2')
print("R2:", r2_score(test_y, predictions))
print("MAE:", mean_absolute_error(test_y, predictions))
print("CV Scores:", cv_scores)

R2: 0.949378799383624
MAE: 0.00136919182692194
CV Scores: [0.94672805 0.81903549 0.83622712 0.89437053 0.86756045]


**18. Count samples per combo**

combo_counts — how many historical rows exist for each country + HS-Short-Name pairing. Some combos have hundreds of rows, others have as few as 1–3. This becomes the basis for filtering out unreliable predictions later.

In [60]:
combo_counts = df.groupby(['country', 'HS-Short-Name']).size()
print(combo_counts.sort_values())

country  HS-Short-Name                   
India    Machines & Mechanical Appliances       1
         Vehicle/Chassis/Tractor Parts          1
Germany  Telecom & Display Equipment            1
         Machines & Mechanical Appliances       2
India    Compressors & Pumps                    2
                                             ... 
China    Electrical Equipment                 335
UAE      Industrial Machinery                 442
China    Industrial Machinery                 568
UAE      General Goods                        938
China    General Goods                       1207
Length: 68, dtype: int64


**19. Average price per kg per combo**

For each combo, the mean of worth-usd ÷ weight-kg. This is what lets the recommender convert a dollar budget into an estimated quantity (kg) later on.

In [61]:
price_per_kg = df.groupby(['country', 'HS-Short-Name']).apply(
    lambda g: (g['worth-usd'] / g['weight-kg']).mean(), include_groups=False
)

**20. Explore reliability thresholds**

Loops through candidate minimum-sample cutoffs (5, 10, 20, 30, 50) and prints how many combos would survive each. This was exploratory — used to decide on a sensible MIN_SAMPLES value rather than picking one arbitrarily.

In [62]:
for threshold in [5, 10, 20, 30, 50]:
    survived = (combo_counts >= threshold).sum()
    print(f"Threshold {threshold}: {survived} of {len(combo_counts)} combos survive")

Threshold 5: 60 of 68 combos survive
Threshold 10: 54 of 68 combos survive
Threshold 20: 40 of 68 combos survive
Threshold 30: 34 of 68 combos survive
Threshold 50: 27 of 68 combos survive


**21. Set MIN_SAMPLES**

Settled on 10 as the minimum number of historical rows a combo needs before the recommender is allowed to suggest it — below that, there isn't enough data to trust a prediction for that combo.

In [63]:
MIN_SAMPLES = 10

**22. Helper: estimate_weight**

One-line utility: budget / price_per_kg. Converts a dollar amount into an estimated quantity in kilograms.

In [64]:
def estimate_weight(budget, ppk):
    return budget / ppk

**23. Helper: get_avg_tax**

Looks up the historical average tariff/VAT for a specific country+product combo. Needed because a hypothetical future purchase doesn't have a real tariff/VAT rate yet — the historical average for that same combo stands in as a reasonable estimate.

In [65]:
def get_avg_tax(df, country, product):
    subset = df[(df['country'] == country) & (df['HS-Short-Name'] == product)]
    return subset['tariff'].mean(), subset['VAT'].mean()

**24. Helper: build_candidate_row**

Packages a country/product/weight/budget/tariff/VAT combination into a single-row DataFrame with column names matching exactly what the model was trained on. pipeline.predict() needs input shaped identically to training data, so this exists purely to keep that formatting consistent everywhere it's needed.

In [66]:
def build_candidate_row(country, product, weight, budget, tariff, vat):
    return pd.DataFrame([{
        'country': country, 'HS-Short-Name': product,
        'weight-kg': weight, 'worth-usd': budget,
        'tariff': tariff, 'VAT': vat
    }])

**25. Helper: matches_any (currently unused — dead code)**

Originally used for fuzzy substring-matching product names when exclusions were typed as free text (e.g. typing "vehicle" to catch multiple vehicle-related categories). Since product/country selection switched to picking numbers from a printed list, filtering now uses exact matches instead — this function isn't called anywhere in the current active code. Safe to delete.

In [67]:
def matches_any(product, terms):
    product_lower = product.lower()
    return any(term.lower() in product_lower for term in terms)


**26. Duplicate redefinitions (redundant — safe to delete)**

Redefines estimate_weight, get_avg_tax, and build_candidate_row a second time, identically to cells 22/24/25. Harmless — the second definition just silently overwrites the first with the same logic — but it's clutter from iterating on the code rather than something intentional.

In [68]:
def estimate_weight(budget, ppk):
    return budget / ppk

def get_avg_tax(df, country, product):
    subset = df[(df['country'] == country) & (df['HS-Short-Name'] == product)]
    return subset['tariff'].mean(), subset['VAT'].mean()

def build_candidate_row(country, product, weight, budget, tariff, vat):
    return pd.DataFrame([{
        'country': country, 'HS-Short-Name': product,
        'weight-kg': weight, 'worth-usd': budget,
        'tariff': tariff, 'VAT': vat
    }])

**27. The recommender engine — get_candidates**

This is the core function. For every country+product combo: skip it if it has too few historical samples (min_samples), skip it if it doesn't match your include_countries/include_products filters or does match exclude_products. For everything that survives: estimate the weight the budget would buy, look up typical tariff/VAT, build a model-ready input row, and call pipeline.predict() to get a real predicted profit rate from the trained Random Forest — not a lookup or approximation. Multiply by budget to get a dollar profit estimate. Returns everything sorted best-profit-first. (This was verified empirically — the model's predictions were checked against known synthetic ground-truth rates and correctly recovered them, and pipeline.predict() was confirmed to fire once per surviving candidate.)

In [69]:
def get_candidates(budget, pipeline, df, price_per_kg, combo_counts, min_samples=MIN_SAMPLES,
                    include_countries=None, include_products=None, exclude_products=None):
    results = []

    for (country, product), count in combo_counts.items():
        if count < min_samples:
            continue
        if include_countries and country not in include_countries:
            continue
        if include_products and product not in include_products:
            continue
        if exclude_products and product in exclude_products:
            continue

        ppk = price_per_kg[(country, product)]
        weight = estimate_weight(budget, ppk)
        tariff, vat = get_avg_tax(df, country, product)
        candidate = build_candidate_row(country, product, weight, budget, tariff, vat)

        predicted_rate = pipeline.predict(candidate)[0]
        predicted_profit = predicted_rate * budget

        results.append({
            'country': country,
            'product': product,
            'estimated_weight_kg': weight,
            'predicted_rate': predicted_rate,
            'predicted_profit_usd': predicted_profit,
            'n_samples': count
        })

    # FIX: Handle empty results gracefully
    if not results:
        print("\n[Warning] No valid combos found matching your criteria.")
        print("Possible reasons: The selected country/product combination doesn't exist,")
        print("or it doesn't have enough samples (>= 10) in the dataset.\n")
        return pd.DataFrame(columns=['country', 'product', 'estimated_weight_kg',
                                     'predicted_rate', 'predicted_profit_usd', 'n_samples'])

    return pd.DataFrame(results).sort_values('predicted_profit_usd', ascending=False).reset_index(drop=True)

**28. allocate_budget**

Takes a set of already-scored candidates (from get_candidates) and splits a budget across them, weighted by each one's predicted rate — a higher-rate combo gets a proportionally bigger slice. Recalculates each one's actual dollar allocation, estimated weight, and profit at that smaller slice specifically. Note: this does not call the model again — it reuses the predicted_rate already computed in get_candidates, which is a reasonable shortcut given profit rate was shown earlier to barely change with shipment size in this dataset.

In [70]:
def allocate_budget(selected, budget, price_per_kg):
    selected = selected.copy()
    weights = selected['predicted_rate'] / selected['predicted_rate'].sum()
    selected['allocated_budget'] = (weights * budget).round(2)
    selected['estimated_weight_kg'] = selected.apply(
        lambda row: estimate_weight(row['allocated_budget'], price_per_kg[(row['country'], row['product'])]),
        axis=1
    ).round(1)
    selected['predicted_profit_usd'] = (selected['allocated_budget'] * selected['predicted_rate']).round(2)
    return selected[['country', 'product', 'allocated_budget', 'estimated_weight_kg', 'predicted_profit_usd', 'n_samples']]

**29. Build the country & product menus**

countries_list/products_list — sorted, deduplicated lists used to print numbered menus so you can select by typing a number instead of an exact name.

In [71]:
countries_list = sorted(df['country'].unique())
products_list = sorted(df['HS-Short-Name'].unique())

**30. Ask for budget**

input() prompt cast to float — the dollar amount the rest of the recommender works from.

In [72]:
budget = float(input("\nEnter your import budget (USD): "))

**31. Ask which products to include/exclude**

Prints the numbered product list, then reads two comma-separated inputs: which product number(s) you want (blank = no restriction) and which to exclude (blank = none). Converts the typed numbers back into the actual product name strings for use in get_candidates.

In [73]:
print("\nProducts:")
for i, p in enumerate(products_list, start=1):
    print(f"{i}. {p}")

include_input = input("\nEnter NUMBER(S) of products you want to import, comma-separated (blank = all): ").strip()
include_products = [products_list[int(i.strip()) - 1] for i in include_input.split(',')] if include_input else None

exclude_input = input("Enter NUMBER(S) to exclude, comma-separated (blank = none): ").strip()
exclude_products = [products_list[int(i.strip()) - 1] for i in exclude_input.split(',')] if exclude_input else None


Products:
1. Articles of Iron & Steel
2. Chemical Products
3. Compressors & Pumps
4. Electrical Equipment
5. General Goods
6. Industrial Machinery
7. Iron & Steel Products
8. Machines & Mechanical Appliances
9. Mineral Fuels & Oils
10. Organic Chemicals
11. Paper & Paperboard
12. Passenger Vehicles
13. Plastics & Polymers
14. Rubber & Tires
15. Semiconductors & Solar Cells
16. Telecom & Display Equipment
17. Vehicle Parts & Accessories
18. Vehicle/Chassis/Tractor Parts


**32. Ask which countries to include**

Same pattern as cell 32, applied to countries — with added validation (isdigit() check and range check) that warns on invalid input instead of crashing, and falls back to "doesn't matter" (None) if nothing valid was entered.

In [74]:
print("\nCountries:")
for i, c in enumerate(countries_list, start=1):
    print(f"{i}. {c}")

country_input = input("\nEnter NUMBER(S) of countries you want to import from, comma-separated "
                      "(blank = doesn't matter): ").strip()

include_countries = None
if country_input:
    include_countries = []
    for i in country_input.split(','):
        i = i.strip()
        if i.isdigit():
            idx = int(i) - 1
            if 0 <= idx < len(countries_list):
                include_countries.append(countries_list[idx])
            else:
                print(f"Warning: Number {i} is out of range (1-{len(countries_list)}). Ignoring.")
        else:
            print(f"Warning: '{i}' is not a valid number. Ignoring.")

    # If all inputs were invalid, revert to None so it doesn't filter anything
    if not include_countries:
        include_countries = None


Countries:
1. China
2. Germany
3. India
4. UAE


**33. Generate & print the ranked candidate list**

Calls get_candidates() once with all your filters, then prints every surviving combo as a numbered list (country, product, estimated weight, predicted profit, sample count). Handles the empty-result case explicitly — if your filters were too restrictive and nothing survived, it says so instead of crashing on an empty table.

In [75]:
candidates = get_candidates(budget, pipeline, df, price_per_kg, combo_counts,
                            include_countries=include_countries,
                            include_products=include_products,
                            exclude_products=exclude_products)

print("\n=== Ranked candidates ===")
if candidates.empty:
    print("No candidates matched your criteria. Try different filters.")
else:
    for i, row in candidates.iterrows():
        print(f"{i + 1}. {row['product']} from {row['country']} — "
              f"est. {row['estimated_weight_kg']:,.1f} kg — predicted profit ${row['predicted_profit_usd']:,.2f} "
              f"({row['n_samples']} samples)")


=== Ranked candidates ===
1. Vehicle/Chassis/Tractor Parts from China — est. 52,157.7 kg — predicted profit $154,744.16 (42 samples)
2. Vehicle/Chassis/Tractor Parts from UAE — est. 42,768.2 kg — predicted profit $154,205.96 (19 samples)
3. Paper & Paperboard from India — est. 281,494.2 kg — predicted profit $135,616.09 (10 samples)
4. Plastics & Polymers from China — est. 95,673.2 kg — predicted profit $134,262.41 (205 samples)
5. Rubber & Tires from China — est. 56,145.5 kg — predicted profit $132,852.00 (78 samples)
6. Plastics & Polymers from India — est. 53,817.4 kg — predicted profit $132,635.76 (46 samples)
7. Rubber & Tires from Germany — est. 48,099.1 kg — predicted profit $132,563.81 (16 samples)
8. Plastics & Polymers from Germany — est. 45,014.3 kg — predicted profit $132,395.82 (30 samples)
9. Plastics & Polymers from UAE — est. 64,557.8 kg — predicted profit $132,245.95 (165 samples)
10. Rubber & Tires from UAE — est. 39,915.4 kg — predicted profit $131,748.87 (67 sample

**34. Let the user pick specific combos, or auto-diversify**

Asks for combo number(s) from the printed list. If you type numbers, allocate_budget runs on exactly those rows. If you leave it blank, it defaults to the top 3 candidates automatically. Either way, the final allocation table and total predicted profit are printed at the end.

In [76]:
if not candidates.empty:
    pick_input = input("\nEnter NUMBER(S) of the combo(s) you actually want to import, comma-separated "
                       "(blank = auto-diversify across top 3): ").strip()

    if pick_input:
        picked_indices = [int(i.strip()) - 1 for i in pick_input.split(',')]
        selected = candidates.iloc[picked_indices]
        final_recommendation = allocate_budget(selected, budget, price_per_kg)
        print("\n=== Your selected allocation ===")
    else:
        final_recommendation = allocate_budget(candidates.head(3), budget, price_per_kg)
        print("\n=== Auto-diversified allocation (top 3) ===")

    print(final_recommendation.to_string(index=False))
    total_profit = final_recommendation['predicted_profit_usd'].sum()
    print(f"\nTotal predicted profit across {len(final_recommendation)} combos: ${total_profit:,.2f}")


=== Auto-diversified allocation (top 3) ===
country                       product  allocated_budget  estimated_weight_kg  predicted_profit_usd  n_samples
  China Vehicle/Chassis/Tractor Parts         139231.60              18155.0              53863.19         42
    UAE Vehicle/Chassis/Tractor Parts         138747.35              14834.9              53489.17         19
  India            Paper & Paperboard         122021.05              85870.5              41370.04         10

Total predicted profit across 3 combos: $148,722.40
